# Ocean sounds: a classifier trained only on generated data

**Question (Ali, Oct 27).** If a classifier never sees a real recording during training, only
clips generated by Path A, how well does it recognise the six classes in real recordings?

**Design.** One small CNN, trained with one fixed recipe on four training sets, tested on real
clips from **fold 5**, which no earlier experiment has touched:

| Training set | What it is | Why it is here |
|---|---|---|
| **Synthetic** | `N_PER_CLASS` clips per class generated by ocean Path A (128 mel bands, 10k steps, EMA, guidance 1.0), turned into audio with Griffin-Lim | The answer to the question. |
| **Real-matched** | the same number of real clips per class, from the training folds | The fair comparison: same amount of data, real instead of generated. |
| **Real-matched via Griffin-Lim** | those real clips through the generator's mel front end and Griffin-Lim, no generator | Separates the generator's cost from the inversion's: Synthetic vs this row is the generator alone. |
| **Real-full** | every real training clip (about 25k), class-balanced draws | The ceiling with all the real data. |

**Frequency range: 10 Hz-1 kHz** for the classifier (Ali, Oct 27). The front-end probe found that
a classifier that hears only 2.5-5 kHz, where there are no fish calls, still recognises vessel and
toadfish (F1 0.99-1.00) from their recording sites. Restricting the classifier to 10 Hz-1 kHz takes
that band away. 128 mel bands over this range need a 2,048-point FFT (with 1,024 points, 30 bands
would duplicate each other); the probe's 20 Hz-1 kHz / FFT 2,048 candidate scored the same as the
full-range front end on the groupers. The **generator is unchanged** (it was trained on 20 Hz-5 kHz):
its audio is analysed with the 10 Hz-1 kHz front end, exactly like real audio. In practice the range
starts at 20 Hz, because every clip is high-passed at 20 Hz when loaded.

**Fixed before running:**
* The recipe (learning rate, steps, SpecAugment) is chosen once, on Real-matched, scored on fold 4,
  then frozen for every training set.
* Headline: **F1 on the four fine-grained classes** (Nassau, yellowfin and black grouper,
  squirrelfish), Synthetic vs Real-matched, with a 95% grouped-bootstrap interval over test
  recording groups. Toadfish and vessel are reported but say little about the calls.
* Fold 5 is scored once at the end. No setting is changed after seeing a fold-5 number.

**Resumable.** Generated clips are saved per class and every condition when it finishes, so a
disconnect costs at most the step in progress. Runtime on a T4: sampling about 30-60 minutes, the
rest about 20-30 minutes. Set `QUICK_TEST = True` first.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import tstr as T

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from IPython.display import Audio, display
from diffusers.training_utils import EMAModel

assert hasattr(C, "load_ocean") and hasattr(T, "train_clf"), "old code: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

## 2. Config

The data settings must match the training notebook (`ocean_path_a.ipynb`): they decide which clips
are in which fold. The recipe grid and the headline are fixed here, before anything is trained.

In [ ]:
QUICK_TEST = False          # True: a few minutes end to end, numbers meaningless

# data and generator: must match notebooks/ocean_path_a.ipynb
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"
GROUP_SIZE  = 20
HIGHPASS_HZ = 20.0
NORMALIZE   = "rms"
BALANCED    = True
OUT_ROOT    = "/content/drive/MyDrive/thesis_runs/ocean"

cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data")      # the generator's config
FE_TAG        = "" if (cfg.n_fft, cfg.f_max) == (1024, None) else f"_fft{cfg.n_fft}_fmax{cfg.f_max:g}"
TAG           = FE_TAG + f"_hp{HIGHPASS_HZ:g}_{NORMALIZE}" + ("_bal" if BALANCED else "")
RUN_DIR       = f"{cfg.out_root}/path_a128{TAG}"
GEN_FRONTEND  = f"{cfg.out_root}/frontend_128mel{FE_TAG}_hp{HIGHPASS_HZ:g}_{NORMALIZE}.json"

# the classifier's front end: 128 mel bands, 10 Hz - 1 kHz, 2,048-point FFT (205 ms window, 4.9 Hz bins)
CLS_FE = dict(n_fft=2048, f_min=10.0, f_max=1000.0)

# synthetic set
N_PER_CLASS = 500           # generated clips per class (and real clips per class for Real-matched)
GUIDANCE    = 1.0
DDIM_STEPS  = None          # None: the training config's inference_steps
GEN_BATCH   = 64            # lower if the GPU runs out of memory
GL_ITERS    = 64

# classifier recipe: chosen once on Real-matched with fold 4, then frozen
GRID = [dict(lr=lr, steps=st, specaug=sa) for lr in (1e-3, 3e-3) for st in (1000, 2000) for sa in (False, True)]
BATCH, N_SEEDS, N_BOOT = 64, 3, 2000
FINE = ["Nassau grouper", "Yellowfin grouper", "Black grouper", "Squirrelfish"]

if QUICK_TEST:
    N_PER_CLASS, DDIM_STEPS, GL_ITERS, N_SEEDS, N_BOOT = 4, 5, 8, 2, 200
    GRID = [dict(lr=3e-3, steps=20, specaug=False), dict(lr=3e-3, steps=20, specaug=True)]

OUT_DIR = Path(f"{cfg.out_root}/synonly/path_a128{TAG}_n{N_PER_CLASS}_g{GUIDANCE:g}" + ("_quick" if QUICK_TEST else ""))
(OUT_DIR / "gen_parts").mkdir(parents=True, exist_ok=True)
C.seed_everything(cfg.seed)
print("generator run:", RUN_DIR); print("results:", OUT_DIR)

## 3. Data

Real clips are center-cropped to the model's clip length (19,812 samples, 1.98 s), the crop the
probe and the evaluation used. Real-matched draws `N_PER_CLASS` clips per class at random from
the training folds (fixed seed). Fold 4 chooses the recipe; fold 5 is the test set.

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
class_names = data["class_names"]; n_classes = len(class_names)
train_split, val_split, test_split = C.split_by_fold(data, cfg)
del data
fine_idx = [class_names.index(n) for n in FINE]

def center_crop(w):
    off = (w.shape[-1] - cfg.clip_samples) // 2
    return w[..., off:off + cfg.clip_samples]

ytr_all = train_split["labels"].numpy()
r = np.random.default_rng(cfg.seed)
matched_idx = []
for c in range(n_classes):
    pool = np.flatnonzero(ytr_all == c)
    if len(pool) < N_PER_CLASS:
        print(f"WARNING: {class_names[c]} has only {len(pool)} training clips; Real-matched uses all of them")
    matched_idx.append(r.choice(pool, min(N_PER_CLASS, len(pool)), replace=False))
matched_idx = torch.from_numpy(np.sort(np.concatenate(matched_idx)))
matched_wave = center_crop(train_split["waveforms"][matched_idx]).float()
matched_y = train_split["labels"][matched_idx]

val_wave, val_y = center_crop(val_split["waveforms"]), val_split["labels"].numpy()
test_wave, test_y = center_crop(test_split["waveforms"]), test_split["labels"].numpy()
test_groups = np.array(test_split["groups"])
print(f"train {len(ytr_all)} | Real-matched {len(matched_y)} | val (fold 4) {len(val_y)} | test (fold 5) {len(test_y)} clips")
print("test clips per class:", np.bincount(test_y, minlength=n_classes).tolist())

## 4. Generate the synthetic set

The ocean Path A model from `RUN_DIR/last.pt` (EMA weights, 10k steps), class by class. Each class
has its own fixed seed and is saved as soon as it is done, so a rerun after a disconnect continues
with the next class.

In [ ]:
gen_fe = C.MelFrontEnd.load(GEN_FRONTEND, cfg, device=device)
parts = {c: OUT_DIR / "gen_parts" / f"class{c}.pt" for c in range(n_classes)}
todo = [c for c, f in parts.items() if not f.exists()]
if todo:
    ck = torch.load(f"{RUN_DIR}/last.pt", map_location="cpu")
    rc = ck["config"]
    for k in ("n_mels", "n_fft", "hop_length", "f_min", "f_max", "spec_frames", "sample_rate", "num_classes"):
        assert rc.get(k) == getattr(cfg, k), f"checkpoint {k} = {rc.get(k)}, this notebook {getattr(cfg, k)}"
    print(f"checkpoint: step {ck['step']} of {rc['total_steps']}")
    unet = C.build_unet2d(cfg, dropout=0.0).to(device)
    ema = EMAModel(unet.parameters(), decay=0.999)
    ema.load_state_dict(ck["ema"]); ema.copy_to(unet.parameters()); unet.eval(); del ck
    fwd = lambda m, x, t, y: m(x, t, class_labels=y).sample
    for c in todo:
        t0 = time.time()
        C.seed_everything(1000 + c)
        x = C.sample(unet, fwd, (1, cfg.n_mels, cfg.spec_frames), cfg, device,
                     labels=torch.full((N_PER_CLASS,), c, dtype=torch.long), guidance=GUIDANCE,
                     steps=DDIM_STEPS, batch_size=GEN_BATCH)
        torch.save({"samples": x.half(), "label": c}, parts[c])
        print(f"{class_names[c]:<20} {N_PER_CLASS} clips in {(time.time() - t0) / 60:.1f} min")
    del unet, ema
gen_mel = torch.cat([torch.load(parts[c])["samples"].float() for c in range(n_classes)])
gen_y = torch.arange(n_classes).repeat_interleave(N_PER_CLASS)
print("synthetic set:", tuple(gen_mel.shape), "| range", f"[{gen_mel.min():.2f}, {gen_mel.max():.2f}]")

## 5. To audio, then to the classifier's front end

Generated spectrograms and the Real-matched clips (for the Griffin-Lim condition) go through the
generator's front end and Griffin-Lim, cached in `OUT_DIR`. Then every set, real or generated, is
analysed by the **same** 10 Hz-1 kHz front end, whose dB window is calibrated on real training
clips only.

In [ ]:
def cached(name, fn):
    p = OUT_DIR / f"{name}.pt"
    if p.exists():
        return torch.load(p)
    v = fn(); torch.save(v, p); return v

gen_wave = cached(f"gen_wave_gl{GL_ITERS}", lambda: C.spec_to_wave(gen_mel.clamp(-1, 1), gen_fe, n_iter=GL_ITERS, seed=cfg.seed))
matched_gl = cached(f"matched_gl{GL_ITERS}", lambda: C.spec_to_wave(
    torch.cat([gen_fe(matched_wave[i:i + 64].to(device)).cpu() for i in range(0, len(matched_wave), 64)]),
    gen_fe, n_iter=GL_ITERS, seed=cfg.seed))

cls_cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data", **CLS_FE)
fit_idx = torch.randperm(len(ytr_all), generator=torch.Generator().manual_seed(0))[:4000]
cls_fe = C.MelFrontEnd(cls_cfg, device=device).fit(center_crop(train_split["waveforms"][fit_idx]).float())
print(f"classifier front end: {cls_cfg.f_min:g}-{cls_cfg.f_max:g} Hz, {cls_cfg.n_mels} bands, FFT {cls_cfg.n_fft}, "
      f"dB window [{cls_fe.db_max - cls_cfg.db_range:.1f}, {cls_fe.db_max:.1f}]")

def to_spec(w, batch=256, dtype=torch.float32):
    return torch.cat([cls_fe(w[i:i + batch].to(device).float()).to(dtype) for i in range(0, len(w), batch)])

X = {"Synthetic": (to_spec(gen_wave), gen_y),
     "Real-matched": (to_spec(matched_wave), matched_y),
     "Real-matched via Griffin-Lim": (to_spec(matched_gl), matched_y)}
X_val, X_test = to_spec(val_wave, dtype=torch.float16), to_spec(test_wave, dtype=torch.float16)

fig, axes = plt.subplots(2, n_classes, figsize=(2.6 * n_classes, 5))
for c in range(n_classes):
    for row, name in enumerate(["Real-matched", "Synthetic"]):
        x, y = X[name]
        axes[row, c].imshow(x[y == c][0, 0].cpu(), origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        axes[row, c].set_title(f"{class_names[c]}\n{name.lower()}", fontsize=7)
plt.suptitle("what the classifier sees: 10 Hz - 1 kHz"); plt.tight_layout(); plt.show()
for c in range(n_classes):
    print(f"{class_names[c]}: real, then generated")
    display(Audio(matched_wave[matched_y == c][0, 0].numpy(), rate=cfg.sample_rate, normalize=True))
    display(Audio(gen_wave[gen_y == c][0, 0].numpy(), rate=cfg.sample_rate, normalize=True))

## 6. The classifier

The probe network used throughout the thesis (`tstr.ProbeCNN`), unchanged, so this result is
directly comparable with the front-end probe and the ESC-50 TSTR pilot. Each choice and why:

| Choice | Reason |
|---|---|
| Four blocks of two 3x3 convolutions + batch norm + ReLU (32, 64, 128, 256 channels), each followed by 2x2 max-pooling; ~1.2M parameters | A standard small spectrogram CNN. The question is about the training data, not the classifier, so a plain, well-understood network that trains in about a minute is the right tool; on real clips it already separates the six classes at 0.96-0.98 F1. |
| Average over time, keep the 8 frequency rows, dropout 0.3, one linear layer | A call can start anywhere in the 2 s window, so time position is averaged away. Frequency position is what separates the groupers, so it is kept. With 10 Hz-1 kHz, each of the 8 rows now covers a slice of the call band instead of the whole 5 kHz range. |
| 128 mel bands, 10 Hz-1 kHz, FFT 2,048 | Ali's range, which removes the band where the site cue was measured. The mel scale gives the lowest frequencies the most bands (43 of 128 below 250 Hz, where the grouper calls peak). FFT 2,048 is the smallest size at which none of the 128 bands duplicate each other at 10 kHz. |
| One recipe for every training set, chosen on Real-matched with fold 4 | If each set were tuned separately, differences would mix data quality with tuning luck. Real-matched has the same size and class balance as the synthetic set, so it is the fair set to tune on. |
| Grid: learning rate {1e-3, 3e-3} x steps {1,000, 2,000} x SpecAugment {off, on} | The settings that matter most for ~3,000 training clips; 8 candidates. Ties go to the simpler recipe (fewer steps, no augmentation, lower learning rate). |
| SpecAugment: one frequency mask (<= 16 bands) and one time mask (<= 24 frames), filled with -1 | Standard regularisation for small spectrogram sets; -1 is "silence" in this front end. |
| AdamW (weight decay 1e-4), one-cycle learning rate, fixed step count, final weights, 3 seeds | No early stopping, so neither fold 4 nor fold 5 influences when training stops; seeds show the run-to-run spread. |
| Real-full: class-balanced draws from all ~25k training clips | Without balancing, toadfish (8.6x black grouper) would dominate; balanced draws match the other sets, which are balanced by construction. |

In [ ]:
def f1_per_class(y, p, k=n_classes):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    prec, rec = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)

@torch.no_grad()
def predict(model, Xs, batch=256):
    return torch.cat([model(Xs[i:i + batch].float()).float().softmax(-1).cpu() for i in range(0, len(Xs), batch)]).numpy()

train_wave_gpu = center_crop(train_split["waveforms"]).to(device)          # float16
ytr_gpu = train_split["labels"].to(device)
w_bal = (1.0 / torch.bincount(train_split["labels"], minlength=n_classes).float())[train_split["labels"]]
def batch_real_full(g, n):
    i = torch.multinomial(w_bal, n, replacement=True, generator=g).to(device)
    return cls_fe(train_wave_gpu[i].float()), ytr_gpu[i]

getters = {"Real-full": batch_real_full, **{n: T.batch_fixed(x, y, device) for n, (x, y) in X.items()}}
print(f"classifier parameters: {sum(p.numel() for p in T.ProbeCNN(n_classes).parameters()):,}")

## 7. Choose the recipe (Real-matched, fold 4), then freeze it

In [ ]:
rec_file = OUT_DIR / "recipe.json"
if rec_file.exists():
    saved = json.load(open(rec_file)); RECIPE, grid = saved["recipe"], pd.DataFrame(saved["grid"])
    print("frozen recipe (saved earlier):", RECIPE)
else:
    rows = []
    for rc in GRID:
        m = T.train_clf(getters["Real-matched"], n_classes, seed=1000, device=device, batch=BATCH, **rc)
        f1 = f1_per_class(val_y, predict(m, X_val).argmax(1))
        rows.append({**rc, "fold-4 macro-F1": f1.mean(), "fold-4 fine-grained F1": f1[fine_idx].mean()})
        print(f"{rc}  fold 4 macro-F1 {f1.mean():.3f} | fine-grained {f1[fine_idx].mean():.3f}")
    grid = pd.DataFrame(rows)
    best = grid.sort_values(["fold-4 macro-F1", "steps", "specaug", "lr"], ascending=[False, True, True, True]).iloc[0]
    RECIPE = dict(lr=float(best["lr"]), steps=int(best["steps"]), specaug=bool(best["specaug"]))
    json.dump({"recipe": RECIPE, "grid": grid.to_dict("records")}, open(rec_file, "w"), indent=2, default=float)
    print("\nfrozen recipe:", RECIPE)

## 8. Train every set with the frozen recipe, test on fold 5

In [ ]:
probs = {}
t0 = time.time()
for name, get in getters.items():
    f = OUT_DIR / f"test_probs_{name.replace(' ', '_')}.npz"
    if f.exists():
        z = np.load(f); probs[name] = [z[k] for k in sorted(z.files)]
    else:
        probs[name] = [predict(T.train_clf(get, n_classes, seed=s, device=device, batch=BATCH, **RECIPE), X_test) for s in range(N_SEEDS)]
        np.savez(f, **{f"seed{s}": p for s, p in enumerate(probs[name])})
    f1s = np.array([f1_per_class(test_y, p.argmax(1)) for p in probs[name]])
    print(f"{name:<30} macro-F1 {f1s.mean(1).mean():.3f} | fine-grained {f1s[:, fine_idx].mean(1).mean():.3f} "
          f"+/- {f1s[:, fine_idx].mean(1).std():.3f}   ({(time.time() - t0) / 60:.1f} min so far)")

rows = []
for name, ps in probs.items():
    f1s = np.array([f1_per_class(test_y, p.argmax(1)) for p in ps])
    rows.append({"training set": name, "macro-F1": f1s.mean(1).mean(), "fine-grained F1": f1s[:, fine_idx].mean(1).mean(),
                 "fine sd": f1s[:, fine_idx].mean(1).std(), **{n: f1s[:, i].mean() for i, n in enumerate(class_names)}})
results = pd.DataFrame(rows).set_index("training set")
results["fine vs Real-matched"] = results["fine-grained F1"] / results.loc["Real-matched", "fine-grained F1"]
results.round(3)

## 9. Headline: Synthetic vs Real-matched

Differences in fine-grained F1 with 95% intervals from a grouped bootstrap: whole test recording
groups are resampled within each class, and every training set is scored on the same resample
(seed-averaged probabilities), so the differences are paired.

In [ ]:
rng = np.random.default_rng(0)
cls_groups = [[np.flatnonzero((test_y == c) & (test_groups == g)) for g in np.unique(test_groups[test_y == c])]
              for c in range(n_classes)]
pred = {n: np.mean(ps, axis=0).argmax(1) for n, ps in probs.items()}
boot = {n: [] for n in pred}
for _ in range(N_BOOT):
    idx = np.concatenate([np.concatenate([grp[j] for j in rng.integers(len(grp), size=len(grp))]) for grp in cls_groups])
    for n, p in pred.items():
        f1 = f1_per_class(test_y[idx], p[idx]); boot[n].append((f1.mean(), f1[fine_idx].mean()))
boot = {n: np.array(v) for n, v in boot.items()}

comparisons = [("Synthetic", "Real-matched"), ("Synthetic", "Real-matched via Griffin-Lim"),
               ("Real-matched via Griffin-Lim", "Real-matched"), ("Real-matched", "Real-full")]
diff_rows = []
for a, b in comparisons:
    d = boot[a] - boot[b]
    diff_rows.append({"comparison": f"{a} - {b}",
                      "fine-grained F1 diff": d[:, 1].mean(), "fine 95% CI": tuple(np.round(np.percentile(d[:, 1], [2.5, 97.5]), 3)),
                      "macro-F1 diff": d[:, 0].mean(), "macro 95% CI": tuple(np.round(np.percentile(d[:, 0], [2.5, 97.5]), 3))})
diffs = pd.DataFrame(diff_rows).set_index("comparison")
display(diffs.round(3))

s, m = results.loc["Synthetic", "fine-grained F1"], results.loc["Real-matched", "fine-grained F1"]
print(f"\nA classifier trained only on generated clips reaches fine-grained F1 {s:.3f} on real fold-5 recordings, "
      f"{s / m:.0%} of the same classifier trained on the same number of real clips ({m:.3f}).")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, name in zip(axes, ["Real-matched", "Real-matched via Griffin-Lim", "Synthetic"]):
    cm = np.bincount(test_y * n_classes + pred[name], minlength=n_classes ** 2).reshape(n_classes, n_classes)
    cm = cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(n_classes), class_names, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(n_classes), class_names, fontsize=7); ax.set_title(f"trained on: {name}", fontsize=9)
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, f"{cm[i, j]:.0%}", ha="center", va="center", color="w", fontsize=7)
plt.suptitle("real fold-5 recordings (rows: true class)"); plt.tight_layout()
plt.savefig(OUT_DIR / "confusion.png", dpi=130); plt.show()

## 10. Save

In [ ]:
out = {"commit": commit, "quick_test": QUICK_TEST, "recipe": RECIPE, "grid": grid.to_dict("records"),
       "classifier_front_end": CLS_FE, "n_per_class": N_PER_CLASS, "guidance": GUIDANCE, "run_dir": RUN_DIR,
       "results": results.round(4).reset_index().to_dict("records"),
       "differences": diffs.reset_index().astype(str).to_dict("records"),
       "per_seed": {n: [f1_per_class(test_y, p.argmax(1)).round(4).tolist() for p in ps] for n, ps in probs.items()}}
with open(OUT_DIR / f"synonly_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved", OUT_DIR / f"synonly_{commit}.json")

## How to read the result

* **The headline is Synthetic vs Real-matched on the fine-grained classes.** It says how much of
  what real recordings teach a classifier, the generated clips teach as well. On ESC-50 the
  equivalent number was 92%.
* **Synthetic vs Real-matched via Griffin-Lim isolates the generator.** If Synthetic is close to
  that row, the remaining gap to real comes from the inversion, not from the generator; if it is
  far below, the generator is missing something the calls need.
* **Expect Synthetic below Real-matched.** The evaluation found the generated clips under-dispersed
  (too typical), so the classifier sees fewer hard examples. Where the confusion matrix loses
  accuracy (for example Nassau vs yellowfin) shows which classes that hurts.
* **Toadfish and vessel** should now be harder than before for every training set, because the
  site cue above 2.5 kHz is gone; whatever separates them below 1 kHz may still include site
  differences, which this design cannot rule out.
* **Fold 5 has now been used.** Treat these numbers as final for this generator and front end;
  changing a setting and rerunning would make fold 5 a validation set.